# Python 数据数学分析核心课件

## 目录

## 1. 引言:从工程代码回归数据本质

### 1.1 为什么是 Python？

在传统的软件后端开发中, 关注点通常是 **状态管理、网络 I/O、并发安全与面向对象的解耦抽象**. 然而, 当进入数据分析与科学计算领域时, 思维模型需要发生根本性转变:

```
[传统后端工程思维]                     [现代数据分析/矩阵思维]
对象封装(OOP)                   ->   数据与操作分离(Data-Oriented Design)
逐个遍历(for item in list)       ->   批量向量化计算(SIMD / Tensor Processing)
复杂控制流(if-else / 状态机)      ->   布尔掩码与索引选择(Mask & Fancy Indexing)
显式多线程/锁竞争                 ->   底层 C/Fortran/BLAS 内存连续并行计算
```

Python 之所以能够统治现代数据科学与 AI 领域, 并非因为其解释器运行效率高(相反, CPython 的解释器开销与 GIL 限制众所周知), 而是因为它的 **"胶水特性" 与 C-API 扩展能力**:

1. 上层提供极具表现力的动态语法和交互式 REPL 环境;
2. 下层无缝对接 OpenBLAS、MKL、CUDA 等底层硬件加速库;
3. 形成了从数据摄取(Pandas/Arrow)、处理(NumPy/SciPy)、可视化(Matplotlib/Seaborn)到模型训练(PyTorch/Scikit-learn)的完整闭环. 

> 为什么 Python 明明什么都能算, 我们还需要 NumPy 和 Pandas？

> NumPy 和 Pandas 真正带来的价值, 是“多了一堆 API”, 还是“改变了数据和计算的抽象方式”？

### 我们真正要学的不是 API

假设现在拿到 200 万条出租车订单. 

我们需要分析:

```text
哪些订单不可信？

什么时候最忙？

什么时候流水最高？

哪些区域最热门？

哪些订单看起来异常？

为什么同一个计算, 有人的代码跑几十秒, 有人的代码只跑几秒？

怎么把这些分析变成公司里以后还能复用的工具？
```

当然, 可以全部使用 Python:

```python
for row in rows:
    ...
```

但问题并不在于 Python **能不能做**. 

真正的问题是:

> **Python 原生的数据结构和逐对象执行模型, 并不是专门针对百万级同构数值计算和二维表格分析设计的. **

但是 Numpy 和 Pandas 是 Python 中专门为数据分析设计的

1. NumPy 的核心是同构 $N$ 维数组 `ndarray`；官方文档将其定义为 NumPy 的核心多维数组结构, 并围绕数组提供高效的数学、逻辑、选择、排序、线性代数等运算. 
2. Pandas 则在数组之上增加了 **标签、索引、缺失值、异构列、自动对齐、分组、连接和时间序列** 等更贴近表格数据分析的抽象；`Index` 本身就是 Pandas 用于索引和对齐的轴标签对象. 

```mermaid
flowchart TD
    A["Python 原生对象<br/>list / dict / Python object"] --> B["NumPy ndarray"]
    B --> C["Pandas Series / DataFrame"]
    C --> D["业务分析"]
    D --> E["性能工程"]
    E --> F["领域工具 / Pipeline / Accessor"]

    B --- B1["dtype / shape / strides"]
    B --- B2["ufunc / vectorization"]
    B --- B3["broadcasting"]

    C --- C1["Index / alignment"]
    C --- C2["groupby / merge"]
    C --- C3["resample / rolling"]
```

一句话概括:
> **NumPy 解决“如何高效计算数组”；Pandas 解决“如何带着业务语义高效组织、查询、组合和分析表格数据”. **

### 出租车数据集简介

NYC TLC Yellow Taxi Trip Records 是纽约市出租车与豪华轿车委员会(NYC Taxi & Limousine Commission, TLC)公开发布的纽约市黄色出租车行程记录数据.

### 贯穿本节的七个问题

| 问题 | 业务问题 | 逐步引出的技术 |
|---|---|---|
| **Q1** | 这 200 万条数据可靠吗？ | `dtype`、缺失值、mask、`loc`、`query` |
| **Q2** | 一天中什么时候最忙？ | `datetime`、`groupby`、`agg` |
| **Q3** | 什么时间段的流水和单位运营时间收入最高？ | 向量化、`assign`、`groupby`、派生指标 |
| **Q4** | 哪些上下车区域最热门？ | `value_counts`、`merge`、连接校验 |
| **Q5** | 哪些订单和时间窗口可能异常？ | `where`、`select`、`quantile`、`transform`、`resample`、`rolling` |
| **Q6** | 为什么同一个分析可以相差一个数量级甚至更多？ | `Python loop`、`apply`、`NumPy`、`Numba`、`eval/query`、内存优化 |
| **Q7** | 如何把分析脚本变成公司可以重复使用的工具？ | `pipe`、纯函数、`Accessor`、领域 API |

### NumPy / Pandas 为什么存在

#### 问题

假设我们有 200 万个金额:

```python
prices = [...]
```

需要:

```text
全部上涨 5%
再增加 3 元服务费
过滤异常价格
求平均值
```

#### 直接使用 Python List 实现

```python
# 直接使用 List 存储与计算
result = []

for price in prices:
    result.append(price * 1.05 + 3)
```

问题是我们把 **200 万次循环控制** 交给了 Python 层. 具体来说, Python 原生 `List` 在这种大批量数值计算场景下存在两大明显短板:

**1. 存储方面**

Python `List` 并不是一块连续的数值缓冲区, 而是一个 **对象引用数组**. 它存储的每一个元素都是一个独立的 Python 对象(这里是 `float`), 每个对象都带有引用计数、类型指针等元信息. 以 200 万个金额为例:

```python
prices = [42.5, 17.3, 58.9, ...]   # 200 万个 float 对象
```

- `List` 本身只存 200 万个**指针**, 每个指针 8 字节(64 位系统)；
- 每一个 `float` 对象本身还要额外占用约 24 字节(CPython 对象头)；
- 于是 `List` 存储 200 万个浮点数的实际内存, 大约是相同规模 `ndarray` 的 **3~5 倍**.

```text
List:
  指针数组(8B × 200万) + 每个 float 对象(24B × 200万)
  ≈ 16 MB + 48 MB = 64 MB 左右

ndarray(float64):
  连续缓冲区(8B × 200万)
  ≈ 16 MB
```

内存翻倍, 还意味着缓存命中率更低、CPU 需要跨越更多随机内存地址.

**2. 计算方面**

```python
result = []
for price in prices:
    result.append(price * 1.05 + 3)
```

这段循环的执行, 每一步都发生在 **Python 解释器层**, 而不是底层机器码层:

- 每次 `for` 迭代都要经历字节码分发(bytecode dispatch)；
- 每次读取 `price` 都要做一次 **拆箱**(unboxing, 从 Python 对象还原成 C double)；
- 每次 `price * 1.05 + 3` 结果要重新 **装箱**(boxing)成新的 `float` 对象；
- 每次 `append` 还要调用方法、检查容量、可能触发列表扩容与内存重分配.

当循环次数上升到 200 万, 这些**解释器级开销被放大 200 万倍**, 成为无可忽视的性能瓶颈. 这也是为什么同样的计算, 纯 Python 循环往往比 NumPy 向量化写法慢一到两个数量级——**真正慢的不是"算", 而是"每次都要让解释器来驱动"**.

#### 使用 Numpy 和 Pandas 实现

NumPy:
```python
result = prices * 1.05 + 3
```

Pandas:

```python
result = (
    df
    .query("fare_amount > 0")
    .groupby("pickup_hour")
    .agg(avg_fare=("fare_amount", "mean"))
)
```

### Numpy 和 Pandas 的特点

**NumPy 解决的痛点(纯数值与张量计算):**
在科学计算场景下, 原生 Python 的 `list` 存储机制会导致严重的内存碎片化(频繁指针解引用)和缓存未命中(Cache Miss), 同时动态类型解释器导致循环极慢.NumPy 的诞生终结了这五大痛点:解决内存空间碎片化、消除解释器动态类型开销、填补多维张量代数的空白、终结切片的高昂拷贝代价, 并打通底层 C/BLAS 和 SIMD 硬件加速生态.

**Pandas 解决的痛点(异构业务数据与关系代数):**
真实世界的业务数据不是纯数学矩阵, 而是包含字符串、时间戳、布尔值的异构表格, 常存在缺失值和错位情况.Pandas 解决了:
1. 如何容纳异构数据类型(字符串、数值混合).
2. 在连接表或对齐时如何依据业务“键(Label)”而非纯“位置(Index)”进行运算.
3. 如何像 SQL 一样使用 GroupBy、Join 等关系代数算子.
4. 如何优雅且不中断程序地处理 NaN 缺失数据.

#### 核心设计哲学

##### NumPy 的灵魂核心:`ndarray`(N-dimensional array)
`ndarray`(N-Dimensional Array) 的本质是“同构多维连续内存缓冲区的视图抽象(N-dimensional Array / Tensor)”, 矩阵(Matrix)是它在 2 维空间下的一个特例.它的设计哲学是:**在动态语言中实现静态语言级别的内存布局.**

* **元数据解耦与视图(View):** `ndarray` 将数据的 `Header`(包含 shape、dtype、strides)与底层的 `Data Buffer`(连续物理内存)彻底分离.这使得切片、转置、Reshape 都只需要极小开销修改 Header 且不需要复制物理内存.
* **齐次强类型与 SIMD:** 数据块内类型必须一致, 使得 CPU 能够跨步寻址(索引 × 大小), 并直接投喂给硬件向量化指令.
* **面向数组编程:** 把底层长循环隐式推给 C 语言层面, 上层仅做“张量形态”的声明式表达.

##### Pandas 的根本基石:`Series` 和 `DataFrame`
如果 NumPy 是“同质数学张量”, Pandas 则是“带标签的异构列式数据框”.
* **列式存储架构:** DataFrame 本质是多个一维 `Series` 的字典集合.在底层, 同类型的列会被组织为内存连续的块.列式存储使得“求整列均值”等分析操作在物理上读取连续内存, 极其高效.
* **标签驱动(Label-Based Alignment):** 与 NumPy 严格的矩阵位置形状对齐不同, Series 和 DataFrame 的核心哲学在于“业务主键(Index)绑定”.两张表相加时, Pandas 会自动依据相同的 Index 标签进行匹配(如果没有则视为 NaN), 哪怕它们数据行数或顺序完全不同.这规避了业务分析中最致命的错位问题.
